In [ ]:
# Cell 1 — Setup + hourly Binance pool + CoinGecko mcap (Stage 1)
# Spec deviation: ranking uses CoinGecko historical mcap (spec: CoinMarketCap).
# Monthly PIT top-50 + pair selection happens in Cell 2 (strategy on self).

import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

from backtest.data import DataPanel, FieldSpec

# ── Config ────────────────────────────────────────────────────────────────────
START_UTC = pd.Timestamp("2019-01-01", tz="UTC")
END_UTC = pd.Timestamp.now(tz="UTC").floor("h")
MATIC_TO_POL_DATE = pd.Timestamp("2024-09-04", tz="UTC")

CG_POOL_SIZE = 100          # fetch extra; stables/wrapped stripped → ~80 Binance matches
CG_SLEEP = 1.2              # between CG list pages
CG_MCAP_SLEEP = 6.0         # between per-coin mcap history pulls

CACHE_DIR = Path("data")
CACHE_DIR.mkdir(exist_ok=True)
BINANCE_CACHE = CACHE_DIR / "binance_hourly_top50_pool.parquet"
MCAP_CACHE_DIR = CACHE_DIR / "cg_mcap"
MCAP_CACHE_DIR.mkdir(exist_ok=True)

BINANCE_KLINES = "https://api.binance.com/api/v3/klines"
BINANCE_INFO = "https://api.binance.com/api/v3/exchangeInfo"
CG_MARKETS = "https://api.coingecko.com/api/v3/coins/markets"
CG_CHART = "https://api.coingecko.com/api/v3/coins/{id}/market_chart/range"

KLINES_COLS = [
    "open_time", "open", "high", "low", "close", "base_volume", "close_time",
    "quote_volume", "n_trades", "taker_buy_base", "taker_buy_quote", "ignored",
]

STABLECOINS = frozenset({
    "tether", "usd-coin", "binance-usd", "dai", "true-usd", "frax", "usdd",
    "tusd", "usdp", "gusd", "paxos-standard", "liquity-usd", "fei-usd",
    "wrapped-steth", "staked-ether", "wrapped-bitcoin", "wrapped-ether",
    "terrausd", "terra-luna", "paypal-usd", "euro-coin",
})

# Anchor symbols so delisted-from-top-100 names still have history (Hurst paper + spec generalization)
ANCHOR_BINANCE = [
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT",
    "DOGEUSDT", "ADAUSDT", "SHIBUSDT", "AVAXUSDT", "DOTUSDT",
    "LINKUSDT", "TRXUSDT", "BCHUSDT", "NEARUSDT", "MATICUSDT",
    "UNIUSDT", "LTCUSDT", "ICPUSDT", "ETCUSDT", "HBARUSDT",
    "ATOMUSDT", "FILUSDT", "APTUSDT", "ARBUSDT", "OPUSDT",
    "INJUSDT", "SUIUSDT", "SEIUSDT", "TIAUSDT", "LDOUSDT",
]


def fetch_binance_usdt_set(session=None):
    sess = session or requests.Session()
    r = sess.get(BINANCE_INFO, timeout=30)
    r.raise_for_status()
    out = set()
    for s in r.json()["symbols"]:
        if s.get("quoteAsset") != "USDT" or s.get("status") != "TRADING":
            continue
        if s.get("isSpotTradingAllowed") is False:
            continue
        out.add(s["symbol"])
    return out


def cg_top_coins(n, session=None):
    sess = session or requests.Session()
    result, page = [], 1
    while len(result) < n + 100:
        r = sess.get(
            CG_MARKETS,
            params={
                "vs_currency": "usd",
                "order": "market_cap_desc",
                "per_page": 250,
                "page": page,
                "sparkline": False,
            },
            timeout=30,
        )
        if r.status_code == 429:
            time.sleep(60)
            continue
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        result.extend(batch)
        if len(batch) < 250:
            break
        page += 1
        time.sleep(CG_SLEEP)
    seen, rows = set(), []
    for c in result:
        sym = c["symbol"].upper()
        if c["id"] in STABLECOINS or sym in seen:
            continue
        seen.add(sym)
        rows.append((c["id"], sym))
    return rows[:n]


def build_symbol_pool(binance_usdt):
    cg_rows = cg_top_coins(CG_POOL_SIZE)
    pool = {}  # binance_symbol -> cg_id
    for cg_id, sym in cg_rows:
        b = f"{sym}USDT"
        if b in binance_usdt:
            pool[b] = cg_id
    for b in ANCHOR_BINANCE:
        if b in binance_usdt and b not in pool:
            pool[b] = None  # anchor only; mcap fetch may fail
    if "MATICUSDT" in pool:
        pool.setdefault("POLUSDT", pool["MATICUSDT"])
    return pool


def fetch_klines(symbol, start_ms, end_ms, session=None):
    sess = session or requests.Session()
    out, cur = [], start_ms
    while cur <= end_ms:
        params = dict(symbol=symbol, interval="1h", startTime=cur, endTime=end_ms, limit=1000)
        for attempt in range(5):
            r = sess.get(BINANCE_KLINES, params=params, timeout=30)
            if r.status_code == 200:
                break
            if r.status_code in (429, 418):
                time.sleep(2 ** attempt)
                continue
            if r.status_code == 400:
                return pd.DataFrame(columns=KLINES_COLS).set_index("open_time")
            r.raise_for_status()
        else:
            raise RuntimeError(f"{symbol}: too many retries at {cur}")
        rows = r.json()
        if not rows:
            break
        out.extend(rows)
        last_open = rows[-1][0]
        if len(rows) < 1000:
            break
        cur = last_open + 3_600_000
        time.sleep(0.12)
    if not out:
        return pd.DataFrame(columns=KLINES_COLS).set_index("open_time")
    df = pd.DataFrame(out, columns=KLINES_COLS)
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    for c in ["open", "high", "low", "close", "base_volume", "quote_volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.set_index("open_time").sort_index()[~df.index.duplicated()]


def fetch_cg_mcap(cg_id, start_ts, end_ts, cache_dir, sleep_sec):
    if not cg_id:
        return pd.Series(dtype=float)
    cache_path = cache_dir / f"{cg_id}.parquet"
    end_pd = pd.Timestamp(end_ts)
    if cache_path.exists():
        s = pd.read_parquet(cache_path)["mcap"]
        s.index = pd.to_datetime(s.index, utc=True)
        if len(s) and s.index[-1] >= end_pd - pd.Timedelta(days=3):
            return s
    url = CG_CHART.format(id=cg_id)
    params = {
        "vs_currency": "usd",
        "from": int(pd.Timestamp(start_ts).timestamp()),
        "to": int(end_pd.timestamp()),
    }
    for attempt in range(4):
        try:
            r = requests.get(url, params=params, timeout=60)
            if r.status_code == 429:
                time.sleep(90)
                continue
            r.raise_for_status()
            data = r.json()
            break
        except Exception:
            time.sleep(15 * (attempt + 1))
    else:
        return pd.Series(dtype=float)
    caps = data.get("market_caps") or []
    if not caps:
        return pd.Series(dtype=float)
    m = pd.DataFrame(caps, columns=["ts", "mcap"])
    s = pd.Series(
        m["mcap"].values,
        index=pd.to_datetime(m["ts"], unit="ms", utc=True),
        name=cg_id,
    )
    s = s[~s.index.duplicated(keep="last")].sort_index()
    s.to_frame("mcap").to_parquet(cache_path)
    time.sleep(sleep_sec)
    return s


# ── 1) Resolve Binance symbol pool ───────────────────────────────────────────
session = requests.Session()
binance_usdt = fetch_binance_usdt_set(session)
symbol_pool = build_symbol_pool(binance_usdt)  # binance_sym -> cg_id
binance_symbols = sorted(symbol_pool.keys())
print(f"Binance USDT pool: {len(binance_symbols)} symbols")
print(f"  sample: {binance_symbols[:8]} ... {binance_symbols[-3:]}")

# ── 2) Hourly klines (incremental cache) ───────────────────────────────────────
start_ms = int(START_UTC.timestamp() * 1000)
end_ms = int(END_UTC.timestamp() * 1000)

LEGACY_CACHE = CACHE_DIR / "binance_hourly.parquet"

if BINANCE_CACHE.exists():
    raw = pd.read_parquet(BINANCE_CACHE)
    raw["open_time"] = pd.to_datetime(raw["open_time"], utc=True)
    have = set(raw["symbol"].unique())
    print(f"Loaded Binance cache {BINANCE_CACHE.name}: {len(have)} symbols, {len(raw):,} rows")
elif LEGACY_CACHE.exists():
    raw = pd.read_parquet(LEGACY_CACHE)
    raw["open_time"] = pd.to_datetime(raw["open_time"], utc=True)
    have = set(raw["symbol"].unique())
    print(f"Seeded from legacy {LEGACY_CACHE.name}: {len(have)} symbols -> will extend pool")
else:
    raw = pd.DataFrame(columns=["open_time", "symbol", "close", "quote_volume"])
    have = set()

need = [s for s in binance_symbols if s not in have]
if need:
    print(f"Fetching {len(need)} new/ missing symbols from Binance (1h klines) ...")
    pieces = []
    for i, sym in enumerate(need, 1):
        t0 = time.time()
        df = fetch_klines(sym, start_ms, end_ms, session=session)
        print(f"  [{i:3d}/{len(need)}] {sym:12s} bars={len(df):7,d}  {time.time()-t0:5.1f}s")
        if df.empty:
            continue
        pieces.append(
            df[["close", "quote_volume"]]
            .reset_index()
            .rename(columns={"open_time": "open_time"})
            .assign(symbol=sym)
        )
    if pieces:
        raw = pd.concat([raw, pd.concat(pieces, ignore_index=True)], ignore_index=True)
    raw.to_parquet(BINANCE_CACHE)
    print(f"Cached -> {BINANCE_CACHE}")

# Stitch MATIC + POL → MATICUSDT column
matic = raw[raw["symbol"] == "MATICUSDT"]
pol = raw[raw["symbol"] == "POLUSDT"]
if not matic.empty and not pol.empty:
    matic_keep = matic[matic["open_time"] < MATIC_TO_POL_DATE]
    pol_keep = pol[pol["open_time"] >= MATIC_TO_POL_DATE]
    stitched = pd.concat([matic_keep, pol_keep], ignore_index=True).assign(symbol="MATICUSDT")
    raw_x = pd.concat(
        [raw[~raw["symbol"].isin(["MATICUSDT", "POLUSDT"])], stitched],
        ignore_index=True,
    )
else:
    raw_x = raw.copy()

trade_cols = sorted({s for s in binance_symbols if s != "POLUSDT"})
close = raw_x.pivot(index="open_time", columns="symbol", values="close").reindex(columns=trade_cols)
qvol = raw_x.pivot(index="open_time", columns="symbol", values="quote_volume").reindex(columns=trade_cols)

full_idx = pd.date_range(close.index.min(), close.index.max(), freq="1h", tz="UTC")
close = close.reindex(full_idx)
qvol = qvol.reindex(full_idx)
close.index = close.index.tz_convert("UTC").tz_localize(None)
qvol.index = qvol.index.tz_convert("UTC").tz_localize(None)

# ── 3) CoinGecko mcap → hourly (ffill), column = Binance symbol ───────────────
print(f"\nFetching CoinGecko mcap history for {sum(1 for v in symbol_pool.values() if v)} coins ...")
mcap_cols = {}
for i, (bsym, cg_id) in enumerate(symbol_pool.items(), 1):
    if bsym == "POLUSDT" or not cg_id:
        continue
    t0 = time.time()
    s = fetch_cg_mcap(cg_id, START_UTC, END_UTC, MCAP_CACHE_DIR, CG_MCAP_SLEEP)
    if s.empty:
        print(f"  [{i:3d}] {bsym:12s} mcap=EMPTY")
        continue
    hourly = s.reindex(close.index, method="ffill")
    mcap_cols[bsym] = hourly
    print(f"  [{i:3d}] {bsym:12s} mcap_pts={len(s):6,d}  hourly={hourly.notna().sum():7,d}  {time.time()-t0:5.1f}s")

mcap = pd.DataFrame(mcap_cols).reindex(index=close.index, columns=close.columns)

# ── 4) DataPanel ──────────────────────────────────────────────────────────────
specs = {
    "prices": FieldSpec(lag=0),
    "volume": FieldSpec(lag=0),
    "mcap": FieldSpec(lag=1, missing="ffill", max_staleness=48),  # ~2d stale cap
}

panel = DataPanel(
    prices=close,
    volume=qvol,
    features={"mcap": mcap},
    specs=specs,
    check_outliers=True,
)

print(f"\n── DataPanel (hurst-optimal-exit-pairs Cell 1) ──")
print(f"Bars       : {len(panel.dates):,}")
print(f"Assets     : {len(panel.assets_all)}")
print(f"Range      : {panel.dates[0]} -> {panel.dates[-1]}")
print(f"NaN prices : {close.isna().mean().mean():.2%}")
print(f"NaN volume : {qvol.isna().mean().mean():.2%}")
print(f"NaN mcap   : {mcap.isna().mean().mean():.2%}")
print(f"Fields     : {panel.available_fields()}")
print(f"\nMonthly PIT top-50 will use feature 'mcap' in Cell 2 (strategy).")
print("\nclose.tail(2):")
print(close.tail(2))
print("\nmcap.tail(2) [BTC, ETH]:")
cols = [c for c in ["BTCUSDT", "ETHUSDT"] if c in mcap.columns]
print(mcap[cols].tail(2))

Binance USDT pool: 65 symbols
  sample: ['AAVEUSDT', 'ADAUSDT', 'ALGOUSDT', 'APTUSDT', 'ARBUSDT', 'ASTERUSDT', 'ATOMUSDT', 'AVAXUSDT'] ... ['XLMUSDT', 'XRPUSDT', 'ZECUSDT']
Loaded Binance cache binance_hourly_top50_pool.parquet: 66 symbols, 2,352,473 rows

Fetching CoinGecko mcap history for 60 coins ...
  [  1] BTCUSDT      mcap=EMPTY
  [  2] ETHUSDT      mcap=EMPTY
  [  3] BNBUSDT      mcap=EMPTY
  [  4] XRPUSDT      mcap=EMPTY
  [  5] SOLUSDT      mcap=EMPTY
  [  6] TRXUSDT      mcap=EMPTY
  [  7] DOGEUSDT     mcap=EMPTY
  [  8] ZECUSDT      mcap=EMPTY
  [  9] USDSUSDT     mcap=EMPTY
  [ 10] ADAUSDT      mcap=EMPTY
  [ 11] BCHUSDT      mcap=EMPTY
  [ 12] LINKUSDT     mcap=EMPTY
  [ 13] TONUSDT      mcap=EMPTY
  [ 14] XLMUSDT      mcap=EMPTY
  [ 15] USD1USDT     mcap=EMPTY
  [ 16] USDEUSDT     mcap=EMPTY
  [ 17] SUIUSDT      mcap=EMPTY
  [ 18] LTCUSDT      mcap=EMPTY
  [ 19] AVAXUSDT     mcap=EMPTY
  [ 20] HBARUSDT     mcap=EMPTY
  [ 21] SHIBUSDT     mcap=EMPTY
  [ 22] TAOUSDT      m

C:\Users\User\backtest_engine\backtest\data\panel.py:172: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  rets = self._prices.pct_change()



── DataPanel (hurst-optimal-exit-pairs Cell 1) ──
Bars       : 64,739
Assets     : 64
Range      : 2019-01-01 00:00:00 -> 2026-05-21 10:00:00
NaN prices : 44.71%
NaN volume : 44.71%
NaN mcap   : 100.00%
Fields     : ['prices', 'volume', 'mcap']

Monthly PIT top-50 will use feature 'mcap' in Cell 2 (strategy).

close.tail(2):
symbol               AAVEUSDT  ADAUSDT  ALGOUSDT  APTUSDT  ARBUSDT  ASTERUSDT  \
2026-05-21 09:00:00     88.76      NaN    0.1155    0.959   0.1101      0.727   
2026-05-21 10:00:00     88.17      NaN    0.1153    0.950   0.1090      0.709   

symbol               ATOMUSDT  AVAXUSDT  BCHUSDT  BFUSDUSDT  ...  USDEUSDT  \
2026-05-21 09:00:00     2.057       NaN      NaN     0.9994  ...    1.0003   
2026-05-21 10:00:00     2.039       NaN      NaN     0.9995  ...    0.9997   

symbol               USDSUSDT   UUSDT   VETUSDT  WLDUSDT  WLFIUSDT  XAUTUSDT  \
2026-05-21 09:00:00    1.0005  1.0007  0.006588   0.2593    0.0619   4526.44   
2026-05-21 10:00:00    1.0005  1.

C:\Users\User\AppData\Local\Temp\ipykernel_20292\775151863.py:299: UserWarning: DataPanel: 17658 bar(s) with |return - median| > 10.0*MAD detected.
  Top examples:
    2025-10-10 USDEUSDT: ret=-6.35%, mad_score=635.5
    2025-10-10 USDEUSDT: ret=+6.05%, mad_score=605.9
    2024-04-13 PAXGUSDT: ret=+19.83%, mad_score=200.8
    2021-01-29 DOGEUSDT: ret=+70.02%, mad_score=170.2
    2021-07-24 DEXEUSDT: ret=+69.51%, mad_score=149.2
  Verify these are real moves, not bad ticks. Pass check_outliers=False to silence.
  panel = DataPanel(


In [ ]:
# Cell 1b — Rebuild mcap: Coin Metrics CapMrktCurUSD + dollar-volume fallback (run after Cell 1)
# Why: free CoinGecko only allows last 365d (401 on 2019 windows). CM community has long daily mcap
# for major assets. Unmapped Binance alts use 30d median quote_volume as rank proxy (documented).
# Needs from Cell 1: close, qvol, symbol_pool, START_UTC, END_UTC, MCAP_CACHE_DIR, FieldSpec, DataPanel

import subprocess
import sys
import time

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "coinmetrics-api-client"])
from coinmetrics.api_client import CoinMetricsClient

CM_HOST = "community-api.coinmetrics.io"
CM_METRIC = "CapMrktCurUSD"
CM_SLEEP = 0.2
CM_TIMEOUT = 15
MCAP_DAILY_CACHE = MCAP_CACHE_DIR / "cm_cap_mrkt_daily.parquet"

# Binance stable / synthetic tickers — exclude from mcap ranking (not in top-50 crypto)
BINANCE_MCAP_EXCLUDE = frozenset({
    "USDTUSDT", "USDCUSDT", "USDSUSDT", "USDEUSDT", "USD1USDT", "BFUSDUSDT",
    "UUSDT", "RLUSDUSDT", "PAXGUSDT", "XAUTUSDT", "EURUSDT", "FDUSDUSDT",
})

# Coin Metrics asset id (not always == strip USDT). See catalog: community-api.coinmetrics.io
CM_ASSET_OVERRIDE = {
    "BTCUSDT": "btc", "ETHUSDT": "eth", "BNBUSDT": "bnb", "SOLUSDT": "sol",
    "XRPUSDT": "xrp", "ADAUSDT": "ada", "DOGEUSDT": "doge", "TRXUSDT": "trx",
    "LINKUSDT": "link", "DOTUSDT": "dot", "AVAXUSDT": "avax", "ATOMUSDT": "atom",
    "LTCUSDT": "ltc", "BCHUSDT": "bch", "NEARUSDT": "near", "UNIUSDT": "uni",
    "ETCUSDT": "etc", "HBARUSDT": "hbar", "ICPUSDT": "icp", "FILUSDT": "fil",
    "APTUSDT": "aptos", "ARBUSDT": "arb", "OPUSDT": "op", "INJUSDT": "inj",
    "SUIUSDT": "sui", "SEIUSDT": "sei", "TIAUSDT": "tia", "LDOUSDT": "ldo",
    "MATICUSDT": "matic", "POLUSDT": "matic", "TONUSDT": "ton", "SHIBUSDT": "shib",
    "AAVEUSDT": "aave", "ALGOUSDT": "algo", "PEPEUSDT": "pepe", "BONKUSDT": "bonk",
    "RENDERUSDT": "rndr", "WLDUSDT": "wld", "ENAUSDT": "ena", "XLMUSDT": "xlm",
    "ZECUSDT": "zec", "DASHUSDT": "dash", "QNTUSDT": "qnt",
}

# Delete empty/bad cache from the broken run (CM OK: 0)
if MCAP_DAILY_CACHE.exists():
    _probe = pd.read_parquet(MCAP_DAILY_CACHE)
    if _probe.empty or _probe.notna().sum().sum() == 0:
        MCAP_DAILY_CACHE.unlink()
        print("Removed empty CM cache from failed run.")


def binance_to_cm_asset(bsym: str) -> str:
    if bsym in CM_ASSET_OVERRIDE:
        return CM_ASSET_OVERRIDE[bsym]
    return bsym.replace("USDT", "").lower()


def _naive_dt_index(idx) -> pd.DatetimeIndex:
    idx = pd.DatetimeIndex(pd.to_datetime(idx))
    if idx.tz is not None:
        idx = idx.tz_localize(None)
    return idx


def _ts_utc(ts):
    t = pd.Timestamp(ts)
    if t.tz is None:
        return t.tz_localize("UTC")
    return t.tz_convert("UTC")


def fetch_cm_mcap_panel(binance_syms, start_ts, end_ts, cache_path):
    """Daily CapMrktCurUSD from Coin Metrics community API. Columns = Binance symbols."""
    start_s = _ts_utc(start_ts).strftime("%Y-%m-%d")
    end_s = _ts_utc(end_ts).strftime("%Y-%m-%d")

    if cache_path.exists():
        daily = pd.read_parquet(cache_path)
        daily.index = _naive_dt_index(daily.index)
        if daily.notna().sum().sum() > 0:
            missing = [s for s in binance_syms if s not in daily.columns]
            if not missing:
                ok = {c for c in daily.columns if daily[c].notna().any()}
                return daily.reindex(columns=binance_syms), ok, []

    cm = CoinMetricsClient(host=CM_HOST)
    frames, cm_ok, cm_fail = {}, [], []
    for i, bsym in enumerate(binance_syms, 1):
        asset = binance_to_cm_asset(bsym)
        try:
            df = cm.get_asset_metrics(
                assets=[asset],
                metrics=[CM_METRIC],
                start_time=start_s,
                end_time=end_s,
                frequency="1d",
            ).to_dataframe()
            if df.empty:
                cm_fail.append((bsym, asset, "empty"))
                continue
            s = df.set_index(_naive_dt_index(df["time"]))[CM_METRIC]
            frames[bsym] = s
            cm_ok.append(bsym)
            if i % 10 == 0:
                print(f"  CM [{i:3d}/{len(binance_syms)}] fetched ...")
        except Exception as e:
            cm_fail.append((bsym, asset, str(e)[:80]))
        time.sleep(CM_SLEEP)

    daily = pd.DataFrame(frames) if frames else pd.DataFrame()
    daily = daily.reindex(columns=binance_syms)
    daily.to_parquet(cache_path)
    return daily, set(cm_ok), cm_fail


# ── Rebuild mcap (CM daily -> hourly; proxy = 30d median quote volume) ────────
rank_syms = sorted(
    s for s in close.columns
    if s not in BINANCE_MCAP_EXCLUDE and s != "POLUSDT"
)
print(f"Rebuilding mcap for {len(rank_syms)} symbols via Coin Metrics {CM_METRIC} ...")
print(f"  Range: {_ts_utc(START_UTC).date()} -> {_ts_utc(END_UTC).date()}")

t0 = time.time()
mcap_daily, cm_ok, cm_fail = fetch_cm_mcap_panel(
    rank_syms, START_UTC, END_UTC, MCAP_DAILY_CACHE
)
print(f"  CM OK: {len(cm_ok)} / {len(rank_syms)}  ({time.time()-t0:.1f}s)")

# Dollar-volume proxy for coins CM does not cover (ranking only; not true mcap)
dv_proxy = qvol.rolling(24 * 30, min_periods=24 * 7).median()

mcap_cols = {}
mcap_from_cm, mcap_from_vol = [], []
for bsym in rank_syms:
    if bsym in cm_ok and bsym in mcap_daily.columns and mcap_daily[bsym].notna().any():
        s = mcap_daily[bsym].dropna()
        mcap_cols[bsym] = s.reindex(close.index, method="ffill")
        mcap_from_cm.append(bsym)
    elif bsym in dv_proxy.columns:
        mcap_cols[bsym] = dv_proxy[bsym]
        mcap_from_vol.append(bsym)

mcap = pd.DataFrame(mcap_cols).reindex(index=close.index, columns=close.columns)
print(f"  True CM mcap: {len(mcap_from_cm)} symbols")
print(f"  Volume-proxy rank only: {len(mcap_from_vol)} symbols")
if cm_fail[:3]:
    print(f"  CM fail sample: {cm_fail[:3]}")

# ── Rebuild DataPanel (prices/volume unchanged) ─────────────────────────────
specs = {
    "prices": FieldSpec(lag=0),
    "volume": FieldSpec(lag=0),
    "mcap": FieldSpec(lag=1, missing="ffill", max_staleness=48),
}

panel = DataPanel(
    prices=close,
    volume=qvol,
    features={"mcap": mcap},
    specs=specs,
    check_outliers=True,
)

print("\n-- DataPanel after Cell 1b --")
print(f"Bars       : {len(panel.dates):,}")
print(f"Assets     : {len(panel.assets_all)}")
print(f"NaN mcap   : {mcap.isna().mean().mean():.2%}")
if "BTCUSDT" in mcap.columns and "BTCUSDT" in cm_ok:
    b = mcap["BTCUSDT"].dropna()
    print(f"BTC mcap (CM): last={b.iloc[-1]:,.0f} USD  (expect billions, not ~1e7 vol proxy)")
elif "BTCUSDT" in mcap.columns:
    print("BTC still on volume-proxy — CM fetch did not land for BTC")
if cm_fail:
    print(f"\nFirst 5 CM failures:")
    for row in cm_fail[:5]:
        print(f"  {row}")

# Smoke: PIT monthly top-50 at one past month-end (uses lagged mcap in panel)
view = panel.as_of(pd.Timestamp("2024-06-01"))
mcap_row = view.feature("mcap").iloc[-1].dropna().sort_values(ascending=False)
print(f"\nPIT smoke 2024-06-01 top-10 by mcap rank (lagged feature):")
print(mcap_row.head(10).to_string())
print(f"\nRegistered deviation: spec=CMC mcap; data=CM CapMrktCurUSD + Binance vol proxy for unmapped alts.")

Removed empty CM cache from failed run.
Rebuilding mcap for 56 symbols via Coin Metrics CapMrktCurUSD ...
  Range: 2019-01-01 -> 2026-05-21
  CM [ 10/56] fetched ...
  CM [ 50/56] fetched ...
  CM OK: 20 / 56  (20.8s)
  True CM mcap: 20 symbols
  Volume-proxy rank only: 36 symbols
  CM fail sample: [('APTUSDT', 'aptos', "Bad Parameter: The request contains invalid parameters. API message:\n{'type': 'b"), ('ARBUSDT', 'arb', "Bad Parameter: The request contains invalid parameters. API message:\n{'type': 'b"), ('ASTERUSDT', 'aster', "Bad Parameter: The request contains invalid parameters. API message:\n{'type': 'b")]


C:\Users\User\backtest_engine\backtest\data\panel.py:172: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  rets = self._prices.pct_change()



-- DataPanel after Cell 1b --
Bars       : 64,739
Assets     : 64
NaN mcap   : 45.69%
BTC mcap (CM): last=1,550,639,385,412 USD  (expect billions, not ~1e7 vol proxy)

First 5 CM failures:
  ('APTUSDT', 'aptos', "Bad Parameter: The request contains invalid parameters. API message:\n{'type': 'b")
  ('ARBUSDT', 'arb', "Bad Parameter: The request contains invalid parameters. API message:\n{'type': 'b")
  ('ASTERUSDT', 'aster', "Bad Parameter: The request contains invalid parameters. API message:\n{'type': 'b")
  ('ATOMUSDT', 'atom', "Bad Parameter: The request contains invalid parameters. API message:\n{'type': 'b")
  ('AVAXUSDT', 'avax', "Bad Parameter: The request contains invalid parameters. API message:\n{'type': 'b")

PIT smoke 2024-06-01 top-10 by mcap rank (lagged feature):
symbol
BTCUSDT     1327745507331.984131
ETHUSDT      451203485747.259644
XRPUSDT       51743078101.373726
DOGEUSDT      22961673643.202534
LINKUSDT        18393556537.8876
ADAUSDT       15635360265.805016
XLMUS

C:\Users\User\AppData\Local\Temp\ipykernel_20292\3607790676.py:153: UserWarning: DataPanel: 17658 bar(s) with |return - median| > 10.0*MAD detected.
  Top examples:
    2025-10-10 USDEUSDT: ret=-6.35%, mad_score=635.5
    2025-10-10 USDEUSDT: ret=+6.05%, mad_score=605.9
    2024-04-13 PAXGUSDT: ret=+19.83%, mad_score=200.8
    2021-01-29 DOGEUSDT: ret=+70.02%, mad_score=170.2
    2021-07-24 DEXEUSDT: ret=+69.51%, mad_score=149.2
  Verify these are real moves, not bad ticks. Pass check_outliers=False to silence.
  panel = DataPanel(


In [ ]:
# Cell 2 — HurstOptimalExitPairsStrategy (Stage 2)
# Spec: hourly scan; monthly PIT top-50 + top-20 coint pairs; O-U exits (100k paths).
# Speed: (1) one OU simulate per entry + 400 grid on pi_paths
#        (2) Numba JIT on OU paths + Hurst (optional parallel paths)
#        (3) joblib pool on monthly ADF pair screen (~1225 tests)

import subprocess
import sys

try:
    from statsmodels.tsa.stattools import coint
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])
    from statsmodels.tsa.stattools import coint

try:
    from numba import njit, prange
    HAS_NUMBA = True
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numba"])
    from numba import njit, prange
    HAS_NUMBA = True

try:
    from joblib import Parallel, delayed
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "joblib"])
    from joblib import Parallel, delayed

from itertools import combinations
import os

# Use all logical cores for Numba + joblib (set before Numba initializes)
N_CORES = 12
os.environ["NUMBA_NUM_THREADS"] = str(N_CORES)
os.environ.setdefault("OMP_NUM_THREADS", str(N_CORES))
os.environ.setdefault("MKL_NUM_THREADS", str(N_CORES))

import numpy as np
import pandas as pd

from backtest.strategy import Strategy

COINT_JOBS = N_CORES
GRID_JOBS = N_CORES


@njit(cache=True)
def _local_hurst_numba(x, taus, q):
    denom = 0.0
    for i in range(len(x)):
        denom += abs(x[i]) ** q
    denom /= len(x)
    if denom <= 0.0:
        return np.nan
    n_tau = len(taus)
    log_tau = np.empty(n_tau)
    log_k = np.empty(n_tau)
    n_valid = 0
    for ti in range(n_tau):
        tau = taus[ti]
        if tau >= len(x):
            return np.nan
        acc = 0.0
        cnt = 0
        for i in range(len(x) - tau):
            acc += abs(x[i + tau] - x[i]) ** q
            cnt += 1
        if cnt == 0:
            return np.nan
        k = acc / cnt / denom
        if k <= 0.0:
            return np.nan
        log_tau[n_valid] = np.log(tau)
        log_k[n_valid] = np.log(k)
        n_valid += 1
    if n_valid < 2:
        return np.nan
    sx = 0.0
    sy = 0.0
    sxx = 0.0
    sxy = 0.0
    for i in range(n_valid):
        sx += log_tau[i]
        sy += log_k[i]
        sxx += log_tau[i] * log_tau[i]
        sxy += log_tau[i] * log_k[i]
    den = n_valid * sxx - sx * sx
    if abs(den) < 1e-18:
        return np.nan
    slope = (n_valid * sxy - sx * sy) / den
    return slope / q


@njit(parallel=True, cache=True)
def _simulate_pi_paths_numba(phi, sigma, s0, m, n_paths, max_h, seed):
    np.random.seed(seed)
    pi_paths = np.empty((n_paths, max_h + 1))
    for i in prange(n_paths):
        si = s0
        pi_paths[i, 0] = 0.0
        for step in range(1, max_h + 1):
            eps = np.random.randn()
            si = si + (1.0 - phi) * (m - si) + sigma * eps
            pi_paths[i, step] = si - s0
    return pi_paths


@njit(cache=True)
def _pnls_for_thresholds_numba(pi_paths, pi_plus, pi_minus):
    n_paths, n_steps = pi_paths.shape
    max_h = n_steps - 1
    pnls = np.empty(n_paths)
    for i in range(n_paths):
        exit_idx = max_h
        for j in range(1, n_steps):
            pi = pi_paths[i, j]
            if pi >= pi_plus or pi <= pi_minus:
                exit_idx = j
                break
        pnls[i] = pi_paths[i, exit_idx]
    return pnls


def _coint_pvalue(la: np.ndarray, lb: np.ndarray) -> float:
    try:
        _, p, _ = coint(la, lb, trend="c", autolag=None, maxlag=1)
        return float(p) if np.isfinite(p) else np.nan
    except Exception:
        return np.nan


def _ou_grid_sharpe(pi_paths: np.ndarray, pi_plus: float, pi_minus: float):
    """One grid node Sharpe (thread-safe read of pi_paths)."""
    if HAS_NUMBA:
        pnls = _pnls_for_thresholds_numba(pi_paths, pi_plus, pi_minus)
    else:
        hit = (pi_paths >= pi_plus) | (pi_paths <= pi_minus)
        max_h = pi_paths.shape[1] - 1
        ever = hit.any(axis=1)
        first_idx = np.full(pi_paths.shape[0], max_h, dtype=int)
        first_idx[ever] = np.argmax(hit[ever], axis=1)
        pnls = pi_paths[np.arange(pi_paths.shape[0]), first_idx]
    sd = pnls.std(ddof=1)
    if sd < 1e-12:
        return None
    return pi_plus, pi_minus, float(pnls.mean() / sd)

# Same exclusions as Cell 1b (stable / synthetic — not ranked for top-50)
BINANCE_MCAP_EXCLUDE = frozenset({
    "USDTUSDT", "USDCUSDT", "USDSUSDT", "USDEUSDT", "USD1USDT", "BFUSDUSDT",
    "UUSDT", "RLUSDUSDT", "PAXGUSDT", "XAUTUSDT", "EURUSDT", "FDUSDUSDT",
})


class HurstOptimalExitPairsStrategy(Strategy):
    """Hurst entry (mathematics-12-02911) + O-U optimal pi+/pi- exits (Ch.13 integration).

    Engine rebalance: every hour. Monthly on self: top-50 by mcap feature, top-20 pairs by ADF.
    """

    rebalance_frequency = "D"

    def __init__(
        self,
        tw_hours: int = 168,
        hedge_lookback_hours: int = 720,
        hurst_q: int = 1,
        hurst_threshold: float = 0.5,
        entry_lower_std: float = 1.0,
        entry_upper_std: float = 2.0,
        max_hold_hours: int = 72,
        top_universe: int = 50,
        top_k_pairs: int = 20,
        coint_lookback_hours: int = 4320,
        pair_gross_alloc: float = 0.05,
        num_simulated_paths: int = 100_000,
        profit_taking_grid_min: float = 0.5,
        profit_taking_grid_max: float = 10.0,
        profit_taking_grid_step: float = 0.5,
        stop_loss_grid_min: float = -10.0,
        stop_loss_grid_max: float = -0.5,
        stop_loss_grid_step: float = 0.5,
    ):
        self.tw_hours = tw_hours
        self.hedge_lookback_hours = hedge_lookback_hours
        self.hurst_q = hurst_q
        self.hurst_threshold = hurst_threshold
        self.entry_lower_std = entry_lower_std
        self.entry_upper_std = entry_upper_std
        self.max_hold_hours = max_hold_hours
        self.top_universe = top_universe
        self.top_k_pairs = top_k_pairs
        self.coint_lookback_hours = coint_lookback_hours
        self.pair_gross_alloc = pair_gross_alloc
        self.num_simulated_paths = num_simulated_paths

        n_max = max(0, int(np.floor(np.log2(tw_hours))) - 2)
        self._taus = np.array([2 ** n for n in range(n_max + 1)], dtype=int)

        self._pi_plus_grid = np.arange(
            profit_taking_grid_min,
            profit_taking_grid_max + profit_taking_grid_step / 2,
            profit_taking_grid_step,
        )
        self._pi_minus_grid = np.arange(
            stop_loss_grid_min,
            stop_loss_grid_max + stop_loss_grid_step / 2,
            stop_loss_grid_step,
        )

        self._current_month = None
        self._top50_assets: list[str] = []
        self._selected_pairs: list[tuple[str, str]] = []
        self.open_trades: dict = {}
        self.trade_log: list[dict] = []
        self._event_this_bar = False

    def __repr__(self):
        return (
            f"HurstOptimalExitPairsStrategy(tw={self.tw_hours},hl={self.hedge_lookback_hours},"
            f"q={self.hurst_q},Hth={self.hurst_threshold},"
            f"band=({self.entry_lower_std},{self.entry_upper_std}),"
            f"hold={self.max_hold_hours},u50={self.top_universe},k={self.top_k_pairs},"
            f"clb={self.coint_lookback_hours},alloc={self.pair_gross_alloc},"
            f"n_paths={self.num_simulated_paths})"
        )

    def required_data(self):
        return {"prices": None, "mcap": None}

    def _local_hurst(self, x):
        x = np.asarray(x, dtype=np.float64)
        if np.isnan(x).any():
            return np.nan
        if HAS_NUMBA:
            return float(_local_hurst_numba(x, self._taus, self.hurst_q))
        q = self.hurst_q
        denom = np.mean(np.abs(x) ** q)
        if denom <= 0 or not np.isfinite(denom):
            return np.nan
        log_tau, log_k = [], []
        for tau in self._taus:
            if tau >= len(x):
                return np.nan
            incr = np.abs(x[tau:] - x[:-tau]) ** q
            k = incr.mean() / denom
            if k <= 0 or not np.isfinite(k):
                return np.nan
            log_tau.append(np.log(tau))
            log_k.append(np.log(k))
        slope, _ = np.polyfit(np.array(log_tau), np.array(log_k), 1)
        return slope / q

    def _monthly_top50(self, data) -> list[str]:
        mcap_row = data.feature("mcap").iloc[-1]
        mcap_row = mcap_row.dropna()
        mcap_row = mcap_row.drop(
            index=[s for s in mcap_row.index if s in BINANCE_MCAP_EXCLUDE],
            errors="ignore",
        )
        return mcap_row.sort_values(ascending=False).head(self.top_universe).index.tolist()

    def _signal_at_t(self, prices_hist, a, b_sym):
        if pd.isna(prices_hist[a].iloc[-1]) or pd.isna(prices_hist[b_sym].iloc[-1]):
            return None

        n_needed = self.hedge_lookback_hours + self.tw_hours + 2
        sub_all = prices_hist[[a, b_sym]].dropna()
        if len(sub_all) < n_needed:
            return None
        sub = sub_all.iloc[-(n_needed + 200) :]
        n = len(sub)

        log_a = np.log(sub[a].values)
        log_b = np.log(sub[b_sym].values)
        lr_a = np.diff(log_a)
        lr_b = np.diff(log_b)

        vol_a = pd.Series(lr_a).rolling(self.hedge_lookback_hours).std().values
        vol_b = pd.Series(lr_b).rolling(self.hedge_lookback_hours).std().values

        b_at_sub = np.full(n, np.nan)
        if n >= 3:
            with np.errstate(divide="ignore", invalid="ignore"):
                ratio = np.where(vol_b[:-1] > 0, vol_a[:-1] / vol_b[:-1], np.nan)
            b_at_sub[2:] = ratio

        s = log_a - b_at_sub * log_b
        s_t = s[-1]
        s_window = s[-(self.tw_hours + 1) : -1]

        if len(s_window) < self.tw_hours or np.isnan(s_window).any() or not np.isfinite(s_t):
            return None

        m_val = float(s_window.mean())
        std_val = float((s_window - m_val).std(ddof=1))
        h_val = self._local_hurst(s_window)

        if not (np.isfinite(std_val) and np.isfinite(h_val)) or std_val == 0.0:
            return None

        return {
            "s": float(s_t),
            "m": m_val,
            "std": std_val,
            "h": float(h_val),
            "b": float(b_at_sub[-1]),
            # Clean spread path for O-U (exclude NaN warmup from rolling b)
            "s_hist": np.concatenate([s_window.astype(float), np.array([float(s_t)])]),
        }

    def _fit_ou(self, s_hist: np.ndarray, m: float):
        s = np.asarray(s_hist, dtype=float)
        if len(s) < 10 or np.isnan(s).any():
            return None
        y = np.diff(s)
        x = s[:-1] - m
        var_x = np.var(x, ddof=1)
        if var_x < 1e-14:
            return None
        phi = 1.0 + np.cov(y, x, ddof=1)[0, 1] / var_x
        resid = y - (phi - 1.0) * x
        sigma = float(np.std(resid, ddof=1))
        if not np.isfinite(phi) or not np.isfinite(sigma) or sigma <= 0:
            return None
        if not (-1.0 < phi < 1.0):
            return None
        return float(phi), sigma

    def _simulate_pi_paths(self, phi, sigma, s0, m, n_paths, max_h, seed: int):
        if HAS_NUMBA:
            return _simulate_pi_paths_numba(
                phi, sigma, s0, m, n_paths, max_h, seed & 0xFFFFFFFF
            )
        rng = np.random.default_rng(seed)
        s = np.full(n_paths, s0, dtype=np.float64)
        pi_paths = np.empty((n_paths, max_h + 1), dtype=np.float64)
        pi_paths[:, 0] = 0.0
        for step in range(1, max_h + 1):
            eps = rng.standard_normal(n_paths)
            s = s + (1.0 - phi) * (m - s) + sigma * eps
            pi_paths[:, step] = s - s0
        return pi_paths

    def _pnls_for_thresholds(self, pi_paths: np.ndarray, pi_plus: float, pi_minus: float):
        if HAS_NUMBA:
            return _pnls_for_thresholds_numba(pi_paths, pi_plus, pi_minus)
        hit = (pi_paths >= pi_plus) | (pi_paths <= pi_minus)
        max_h = pi_paths.shape[1] - 1
        ever = hit.any(axis=1)
        first_idx = np.full(pi_paths.shape[0], max_h, dtype=int)
        first_idx[ever] = np.argmax(hit[ever], axis=1)
        return pi_paths[np.arange(pi_paths.shape[0]), first_idx]

    def _calibrate_ou_thresholds(self, phi, sigma, s0, m, seed: int):
        """Simulate once, score 400 grid nodes in parallel threads (spec: 100k paths)."""
        pi_paths = self._simulate_pi_paths(
            phi, sigma, s0, m, self.num_simulated_paths, self.max_hold_hours, seed
        )
        grid = [
            (float(pi_p), float(pi_m))
            for pi_p in self._pi_plus_grid
            for pi_m in self._pi_minus_grid
        ]
        if GRID_JOBS > 1:
            results = Parallel(n_jobs=GRID_JOBS, prefer="threads")(
                delayed(_ou_grid_sharpe)(pi_paths, pi_p, pi_m) for pi_p, pi_m in grid
            )
        else:
            results = [_ou_grid_sharpe(pi_paths, pi_p, pi_m) for pi_p, pi_m in grid]

        best_sharpe = -np.inf
        best_plus = best_minus = None
        for res in results:
            if res is None:
                continue
            pi_p, pi_m, sharpe = res
            if sharpe > best_sharpe:
                best_sharpe = sharpe
                best_plus, best_minus = pi_p, pi_m

        if best_plus is None:
            return None
        return best_plus, best_minus, best_sharpe

    @staticmethod
    def _signed_spread_pnl(side: int, pi: float) -> float:
        """Economic spread PnL: sell-pair (+1) wins when spread falls; buy-pair (-1) when it rises."""
        return float(-side * pi)

    def _exit_reason(self, trade: dict, pi_t: float, hours_held: int) -> str:
        if pi_t >= trade["pi_plus"]:
            return "pi_plus"
        if pi_t <= trade["pi_minus"]:
            return "pi_minus"
        if hours_held >= self.max_hold_hours:
            return "timeout"
        return "unknown"

    def _record_closed_trade(
        self, pair, trade: dict, exit_t, s_exit, pi_exit: float, hours_held: int, h_exit
    ):
        reason = self._exit_reason(trade, pi_exit, hours_held)
        signed = self._signed_spread_pnl(trade["side"], pi_exit)
        a, b = pair
        self.trade_log.append(
            {
                "pair": f"{a}/{b}",
                "a": a,
                "b": b,
                "side": trade["side"],
                "side_label": "sell_pair" if trade["side"] == 1 else "buy_pair",
                "entry_t": trade["entry_t"],
                "exit_t": exit_t,
                "hours_held": hours_held,
                "exit_reason": reason,
                "h_entry": trade["h_entry"],
                "h_exit": h_exit,
                "h_bin": round(np.floor(trade["h_entry"] * 10) / 10, 1),
                "s0": trade["s0"],
                "s_exit": s_exit,
                "m0": trade["m0"],
                "std0": trade["std0"],
                "z_entry": (trade["s0"] - trade["m0"]) / trade["std0"] if trade["std0"] else np.nan,
                "pi_exit": pi_exit,
                "signed_spread_pnl": signed,
                "win": signed > 0,
                "pi_plus": trade["pi_plus"],
                "pi_minus": trade["pi_minus"],
                "phi": trade["phi"],
                "sigma": trade["sigma"],
                "ou_sharpe": trade.get("ou_sharpe"),
                "pi_plus_over_std": trade["pi_plus"] / trade["std0"] if trade["std0"] else np.nan,
                "pi_minus_over_std": trade["pi_minus"] / trade["std0"] if trade["std0"] else np.nan,
            }
        )

    def _select_pairs(self, prices_hist: pd.DataFrame, eligible: list[str]):
        cols = [c for c in eligible if c in prices_hist.columns]
        recent = prices_hist[cols].iloc[-self.coint_lookback_hours :]
        cols = [c for c in cols if recent[c].notna().sum() >= self.coint_lookback_hours * 0.9]
        pair_list = list(combinations(cols, 2))

        def _score_pair(ab):
            a, b = ab
            sub = recent[[a, b]].dropna()
            if len(sub) < self.coint_lookback_hours * 0.9:
                return None
            p = _coint_pvalue(np.log(sub[a].values), np.log(sub[b].values))
            if np.isfinite(p):
                return (ab, p)
            return None

        if COINT_JOBS > 1 and len(pair_list) > 30:
            out = Parallel(n_jobs=COINT_JOBS, prefer="threads")(
                delayed(_score_pair)(ab) for ab in pair_list
            )
        else:
            out = [_score_pair(ab) for ab in pair_list]
        scores = [x for x in out if x is not None]
        scores.sort(key=lambda kv: kv[1])
        return [pair for pair, _ in scores[: self.top_k_pairs]]

    def generate_weights(self, data, t):
        self._event_this_bar = False
        prices = data.prices
        weights = pd.Series(0.0, index=prices.columns)

        ts = pd.Timestamp(t)
        ym = (ts.year, ts.month)

        if ym != self._current_month:
            self._current_month = ym
            self._top50_assets = self._monthly_top50(data)
            if len(prices) >= self.coint_lookback_hours and len(self._top50_assets) >= 2:
                self._selected_pairs = self._select_pairs(prices, self._top50_assets)

        if not self._selected_pairs and not self.open_trades:
            return weights

        k_leg = self.pair_gross_alloc / 2.0

        for pair in list(self.open_trades.keys()):
            a, b = pair
            sig = self._signal_at_t(prices, a, b)
            if sig is None:
                trade = self.open_trades[pair]
                side = trade["side"]
                if side == -1:
                    weights[a] += k_leg
                    weights[b] += -k_leg
                else:
                    weights[a] += -k_leg
                    weights[b] += k_leg
                continue

            s_t = sig["s"]
            trade = self.open_trades[pair]
            side = trade["side"]
            pi_t = s_t - trade["s0"]
            hours_held = int((t - trade["entry_t"]) / pd.Timedelta("1h"))

            exit_now = (
                pi_t >= trade["pi_plus"]
                or pi_t <= trade["pi_minus"]
                or hours_held >= self.max_hold_hours
            )

            if exit_now:
                h_exit = sig["h"] if sig is not None else np.nan
                self._record_closed_trade(
                    pair, trade, t, s_t, pi_t, hours_held, h_exit
                )
                del self.open_trades[pair]
                self._event_this_bar = True
            else:
                if side == -1:
                    weights[a] += k_leg
                    weights[b] += -k_leg
                else:
                    weights[a] += -k_leg
                    weights[b] += k_leg

        for pair in self._selected_pairs:
            if pair in self.open_trades:
                continue
            a, b = pair
            sig = self._signal_at_t(prices, a, b)
            if sig is None or sig["h"] >= self.hurst_threshold:
                continue

            s_t, m_t, std_t = sig["s"], sig["m"], sig["std"]
            side = 0
            if (m_t + self.entry_lower_std * std_t) < s_t < (m_t + self.entry_upper_std * std_t):
                side = +1
            elif (m_t - self.entry_upper_std * std_t) < s_t < (m_t - self.entry_lower_std * std_t):
                side = -1
            if side == 0:
                continue

            ou = self._fit_ou(sig["s_hist"], m_t)
            if ou is None:
                continue
            phi, sigma = ou

            seed = hash((a, b, str(t))) % (2**32)
            cal = self._calibrate_ou_thresholds(phi, sigma, s_t, m_t, seed)
            if cal is None:
                continue
            pi_plus, pi_minus, best_sharpe = cal

            self.open_trades[pair] = {
                "entry_t": t,
                "side": side,
                "m0": m_t,
                "std0": std_t,
                "s0": s_t,
                "h_entry": float(sig["h"]),
                "pi_plus": pi_plus,
                "pi_minus": pi_minus,
                "phi": phi,
                "sigma": sigma,
                "ou_sharpe": float(best_sharpe),
            }
            self._event_this_bar = True
            if side == -1:
                weights[a] += k_leg
                weights[b] += -k_leg
            else:
                weights[a] += -k_leg
                weights[b] += k_leg

        return weights

    def apply_risk(self, proposed, state, data):
        equity = state["equity"]
        if not self._event_this_bar and equity > 0:
            proposed = state["positions"] / equity
        return super().apply_risk(proposed, state, data)


strat = HurstOptimalExitPairsStrategy()
print(strat)
try:
    from numba import config as _numba_config
    _numba_threads = _numba_config.NUMBA_NUM_THREADS
except Exception:
    _numba_threads = "?"
print(f"Accel: cores={N_CORES}, numba={HAS_NUMBA}, numba_threads={_numba_threads}, "
      f"coint_jobs={COINT_JOBS}, grid_jobs={GRID_JOBS}")
print("Note: Engine.run() loop is still 1 process; cores used inside OU sim + monthly ADF + grid.")

rb = strat.rebalance_dates(panel.dates)
print(f"\nRebalance bars : {len(rb):,}  (hourly panel -> expect {len(panel.dates):,})")
print(f"tau for Hurst  : {strat._taus.tolist()}")
print(f"O-U grid       : {len(strat._pi_plus_grid)} x {len(strat._pi_minus_grid)} = "
      f"{len(strat._pi_plus_grid) * len(strat._pi_minus_grid)} nodes")
print(f"required_data(): {strat.required_data()}")

rng = np.random.default_rng(0)
noise = rng.standard_normal(168)
rw = np.cumsum(noise)
trend = np.cumsum(np.abs(noise))
print("\nHurst sanity (TW=168):")
print(f"  white noise : H = {strat._local_hurst(noise):.3f}  (expect ~ 0)")
print(f"  random walk : H = {strat._local_hurst(rw):.3f}  (expect ~ 0.5)")
print(f"  trending    : H = {strat._local_hurst(trend):.3f}  (expect > 0.5)")

# Later date -> more names with mcap history (early 2019 has sparse CM/proxy)
test_t = panel.dates[24_000]
test_view = panel.as_of(test_t)
strat._top50_assets = strat._monthly_top50(test_view)
print(f"\nMonthly top-50 smoke at {test_t}: {len(strat._top50_assets)} assets")
print(f"  head: {strat._top50_assets[:8]}")

sig = strat._signal_at_t(test_view.prices, "BTCUSDT", "ETHUSDT")
print(f"\n_signal_at_t (BTC,ETH): h={sig['h']:.3f} s={sig['s']:.4f} m={sig['m']:.4f}" if sig else "\n_signal_at_t: None")

if sig:
    ou = strat._fit_ou(sig["s_hist"], sig["m"])
    print(f"O-U fit: {ou}")
    if ou:
        import time as _time
        t0 = _time.time()
        cal = strat._calibrate_ou_thresholds(ou[0], ou[1], sig["s"], sig["m"], seed=42)
        print(f"O-U calibrate (n_paths={strat.num_simulated_paths}): {cal}  elapsed={_time.time()-t0:.1f}s")

HurstOptimalExitPairsStrategy(tw=168,hl=720,q=1,Hth=0.5,band=(1.0,2.0),hold=72,u50=50,k=20,clb=4320,alloc=0.05,n_paths=100000)
Accel: cores=12, numba=True, numba_threads=16, coint_jobs=12, grid_jobs=12
Note: Engine.run() loop is still 1 process; cores used inside OU sim + monthly ADF + grid.

Rebalance bars : 64,739  (hourly panel -> expect 64,739)
tau for Hurst  : [1, 2, 4, 8, 16, 32]
O-U grid       : 20 x 20 = 400 nodes
required_data(): {'prices': None, 'mcap': None}

Hurst sanity (TW=168):
  white noise : H = 0.018  (expect ~ 0)
  random walk : H = 0.502  (expect ~ 0.5)
  trending    : H = 1.007  (expect > 0.5)

Monthly top-50 smoke at 2021-09-27 00:00:00: 32 assets
  head: ['BTCUSDT', 'ETHUSDT', 'XRPUSDT', 'ADAUSDT', 'DOTUSDT', 'XLMUSDT', 'DOGEUSDT', 'LINKUSDT']

_signal_at_t (BTC,ETH): h=0.506 s=5.0116 m=4.9566
O-U fit: (0.954489447469991, 0.008714576725468907)
O-U calibrate (n_paths=100000): (0.5, -10.0, -1.823276345622883)  elapsed=5.8s


In [ ]:
# Cell 3 — Costs + liquidity (mandatory per PROTOCOL §6)
from backtest.costs import (
    Commission, Spread, MarketImpact, ShortBorrow,
    CompositeCostModel, LiquidityCap,
)

# Hourly crypto: trading_days=8760 for ShortBorrow (default 252 would inflate borrow ~35x)
HOURS_PER_YEAR = 365 * 24
BORROW_ANNUAL_BPS = 1500

costs = CompositeCostModel([
    Commission(bps=10),
    Spread(half_bps=5),
    MarketImpact(k=10, kind="sqrt", adv_lookback=168),
    ShortBorrow(annual_bps=BORROW_ANNUAL_BPS, trading_days=HOURS_PER_YEAR),
])

liq = LiquidityCap(cap_pct=0.10, adv_lookback=168)

print("Cost components :", [type(m).__name__ for m in costs.models])
for m in costs.models:
    if isinstance(m, Commission):
        print(f"  Commission        bps         = {m.bps}")
    elif isinstance(m, Spread):
        print(f"  Spread            half_bps    = {m.half_bps}")
    elif isinstance(m, MarketImpact):
        print(f"  MarketImpact      k={m.k}, kind={m.kind!r}, adv_lookback={m.adv_lookback}")
    elif isinstance(m, ShortBorrow):
        implied_annual_pct = m.daily_rate * HOURS_PER_YEAR * 100
        print(f"  ShortBorrow       per-bar rate={m.daily_rate:.3e}  (annualized = {implied_annual_pct:.2f}%)")

print(f"\nLiquidityCap      cap_pct={liq.cap_pct}, adv_lookback={liq.adv_lookback}")
print(f"Panel has volume  : {panel.has_field('volume')}  (must be True)")

sample_view = panel.as_of(panel.dates[-1])
sample_trades = pd.Series({"BTCUSDT": 10_000.0, "ETHUSDT": -10_000.0}).reindex(panel.assets_all).fillna(0.0)
tc = costs.trade_cost(sample_trades, sample_view)
hc = costs.holding_cost(sample_trades, sample_view)
print(f"\nSmoke test (last bar, +$10k BTC / -$10k ETH):")
print(f"  trade_cost   = ${tc:,.2f}   ({tc / 20_000 * 1e4:.1f} bps of $20k notional)")
print(f"  holding_cost = ${hc:,.4f}   (short ETH leg; annualized = {hc / 10_000 * HOURS_PER_YEAR * 100:.2f}%)")

Cost components : ['Commission', 'Spread', 'MarketImpact', 'ShortBorrow']
  Commission        bps         = 10
  Spread            half_bps    = 5
  MarketImpact      k=10, kind='sqrt', adv_lookback=168
  ShortBorrow       per-bar rate=1.712e-05  (annualized = 15.00%)

LiquidityCap      cap_pct=0.1, adv_lookback=168
Panel has volume  : True  (must be True)

Smoke test (last bar, +$10k BTC / -$10k ETH):
  trade_cost   = $30.00   (15.0 bps of $20k notional)
  holding_cost = $0.1712   (short ETH leg; annualized = 15.00%)


In [ ]:
# Cell 4 — Single-path Engine run (full history from warmup)
import time
from backtest.engine import Engine

WARMUP_BARS = 5400  # coint 4320 + hedge 720 + tw 168 + buffer

engine = Engine(
    panel,
    strat,
    costs=costs,
    liquidity=liq,
    initial_capital=1_000_000.0,
)

print(f"Engine constructed. Full run from {panel.dates[WARMUP_BARS]} -> {panel.dates[-1]}")
print("O-U: 100k paths x1 per entry + 400 grid (simulate-once optimization in Cell 2)")
print("Expect: slower than Hurst-only notebook due to O-U on each entry; still hours not days.\n")

t0 = time.time()
result = engine.run(start=panel.dates[WARMUP_BARS])
elapsed = time.time() - t0
print(f"...done in {elapsed/60:.1f} min.\n")

md = result.metadata
init_cap = md["initial_capital"]
final_eq = result.equity_curve.iloc[-1]
total_ret = final_eq / init_cap - 1

print(f"Bars                 : {len(result.equity_curve):,}")
print(f"Rebalances           : {md['n_rebalances']:,}")
print(f"Date range           : {md['start']} -> {md['end']}")
print(f"Initial capital      : ${init_cap:>14,.0f}")
print(f"Final equity         : ${final_eq:>14,.0f}")
print(f"Total return         : {total_ret:+.2%}")
print(f"Total costs paid     : ${result.costs.sum():>14,.0f}")
print(f"Max |position weight|: {result.weights.abs().max().max():.2%}")
print(f"Max gross exposure   : {result.weights.abs().sum(axis=1).max():.2%}")
print(f"Max net exposure     : {result.weights.sum(axis=1).abs().max():.2%}")
print(f"Open trades at end   : {len(strat.open_trades)}")
print(f"Closed trades logged : {len(strat.trade_log):,}")
print(f"Unfilled (sum |$|)   : ${result.unfilled.abs().sum().sum():>14,.0f}")

TRADE_LOG_PATH = Path("data/hurst_ou_trade_log.parquet")
if strat.trade_log:
    pd.DataFrame(strat.trade_log).to_parquet(TRADE_LOG_PATH, index=False)
    print(f"Trade log saved -> {TRADE_LOG_PATH}")

non_idle = (result.weights.abs().sum(axis=1) > 1e-6).sum()
print(f"Bars with exposure   : {non_idle:,} / {len(result.equity_curve):,}  ({non_idle/len(result.equity_curve):.1%})")

Engine constructed. Full run from 2019-08-14 00:00:00 -> 2026-05-21 10:00:00
O-U: 100k paths x1 per entry + 400 grid (simulate-once optimization in Cell 2)
Expect: slower than Hurst-only notebook due to O-U on each entry; still hours not days.

...done in 537.7 min.

Bars                 : 59,339
Rebalances           : 59,339
Date range           : 2019-08-14 00:00:00 -> 2026-05-21 10:00:00
Initial capital      : $     1,000,000
Final equity         : $       364,021
Total return         : -63.60%
Total costs paid     : $       606,504
Max |position weight|: 20.00%
Max gross exposure   : 95.73%
Max net exposure     : 30.03%
Open trades at end   : 6
Unfilled (sum |$|)   : $   276,996,770
Bars with exposure   : 59,118 / 59,339  (99.6%)


In [ ]:
# Cell 5 — Trade EDA (Hurst bins, exit reasons, O-U vs entry)
# Requires trade_log: re-run Cell 2+4, OR set RUN_SIGNAL_REPLAY=True (strategy only, no engine/costs).

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TRADE_LOG_PATH = Path("data/hurst_ou_trade_log.parquet")
RUN_SIGNAL_REPLAY = False  # True = rebuild log without full Engine (~hours, no $ PnL)

if RUN_SIGNAL_REPLAY:
    print("Signal-only replay (no engine)...")
    strat_eda = HurstOptimalExitPairsStrategy()
    strat_eda.trade_log = []
    dates = panel.dates[WARMUP_BARS:]
    t0 = time.time()
    for i, t in enumerate(dates):
        strat_eda.generate_weights(panel.as_of(t), t)
        if i and i % 5000 == 0:
            print(f"  {i:,}/{len(dates):,} bars  closed={len(strat_eda.trade_log):,}  open={len(strat_eda.open_trades)}")
    # Still-open at end
    t_last = dates[-1]
    view_last = panel.as_of(t_last)
    for pair, trade in list(strat_eda.open_trades.items()):
        a, b = pair
        sig = strat_eda._signal_at_t(view_last.prices, a, b)
        s_exit = sig["s"] if sig else np.nan
        h_exit = sig["h"] if sig else np.nan
        pi_exit = s_exit - trade["s0"] if np.isfinite(s_exit) else np.nan
        hours_held = int((t_last - trade["entry_t"]) / pd.Timedelta("1h"))
        strat_eda._record_closed_trade(pair, trade, t_last, s_exit, pi_exit, hours_held, h_exit)
        strat_eda.trade_log[-1]["exit_reason"] = "open_at_end"
    pd.DataFrame(strat_eda.trade_log).to_parquet(TRADE_LOG_PATH, index=False)
    print(f"Replay done in {(time.time()-t0)/60:.1f} min -> {TRADE_LOG_PATH}")
    trades = pd.DataFrame(strat_eda.trade_log)
elif TRADE_LOG_PATH.exists():
    trades = pd.read_parquet(TRADE_LOG_PATH)
    print(f"Loaded {len(trades):,} closed trades from {TRADE_LOG_PATH}")
elif getattr(strat, "trade_log", None):
    trades = pd.DataFrame(strat.trade_log)
    print(f"Using in-memory trade_log ({len(trades):,} trades)")
else:
    raise RuntimeError(
        "No trade log. Re-run Cell 2 then Cell 4 (saves parquet), or set RUN_SIGNAL_REPLAY=True."
    )

trades["entry_t"] = pd.to_datetime(trades["entry_t"])
trades["exit_t"] = pd.to_datetime(trades["exit_t"])

print("\n=== Overview ===")
print(f"Trades: {len(trades):,}  |  Wins: {trades['win'].sum():,} ({trades['win'].mean():.1%})")
print(f"Mean signed spread PnL: {trades['signed_spread_pnl'].mean():.4f}  (units: log-spread)")
print(trades["exit_reason"].value_counts().to_string())

# --- 1) Hurst at entry (0.1 bins): wins vs losses ---
trades["h_bin"] = (np.floor(trades["h_entry"] * 10) / 10).clip(0, 0.4)

hurst_tbl = (
    trades.groupby("h_bin", observed=True)
    .agg(
        n=("win", "size"),
        wins=("win", "sum"),
        losses=("win", lambda s: (~s).sum()),
        win_rate=("win", "mean"),
        mean_pnl=("signed_spread_pnl", "mean"),
        med_pnl=("signed_spread_pnl", "median"),
    )
    .reset_index()
)
hurst_tbl["loss_rate"] = 1 - hurst_tbl["win_rate"]
print("\n=== 1) By Hurst H at entry (0.1 bins) ===")
display(hurst_tbl)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = hurst_tbl["h_bin"].astype(str)
axes[0].bar(x, hurst_tbl["wins"], label="wins", color="seagreen")
axes[0].bar(x, hurst_tbl["losses"], bottom=hurst_tbl["wins"], label="losses", color="indianred")
axes[0].set_title("Trade count by H bin")
axes[0].set_xlabel("H entry (floor to 0.1)")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(x, hurst_tbl["mean_pnl"], color=np.where(hurst_tbl["mean_pnl"] >= 0, "seagreen", "indianred"))
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_title("Mean signed spread PnL by H bin")
axes[1].set_xlabel("H entry")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

# --- 2) Losing trades: where do losses come from? ---
losers = trades[~trades["win"]].copy()
print("\n=== 2) Losing trades (signed spread PnL < 0) ===")
print(f"Losers: {len(losers):,} / {len(trades):,} ({len(losers)/len(trades):.1%})")
print("\nExit reason (losers only):")
print(losers["exit_reason"].value_counts(normalize=True).mul(100).round(1).astype(str).radd("% ").to_string())

exit_all = trades.groupby("exit_reason").agg(
    n=("win", "size"),
    win_rate=("win", "mean"),
    mean_pnl=("signed_spread_pnl", "mean"),
    med_hours=("hours_held", "median"),
).sort_values("n", ascending=False)
print("\nAll trades by exit reason:")
display(exit_all)

print("\nLosers by side (entry direction):")
print(losers.groupby("side_label")["signed_spread_pnl"].agg(["count", "mean", "median"]).to_string())

print("\nWorst decile pairs (by mean loser PnL):")
pair_loss = losers.groupby("pair")["signed_spread_pnl"].agg(["count", "mean"]).sort_values("mean")
display(pair_loss.head(15))

# --- 3) O-U exits vs entry ---
print("\n=== 3) O-U calibration vs entry / exit ===")
ou = trades.copy()
ou["pi_plus_z"] = ou["pi_plus"] / ou["std0"]
ou["pi_minus_z"] = ou["pi_minus"] / ou["std0"]
ou["hit_plus"] = ou["exit_reason"] == "pi_plus"
ou["hit_minus"] = ou["exit_reason"] == "pi_minus"
ou["hit_timeout"] = ou["exit_reason"] == "timeout"

cols_ou = [
    "phi", "sigma", "ou_sharpe", "pi_plus", "pi_minus", "pi_plus_z", "pi_minus_z",
    "hours_held", "pi_exit", "signed_spread_pnl", "exit_reason",
]
print("Means by exit reason:")
display(ou.groupby("exit_reason")[cols_ou].mean(numeric_only=True).round(4))

print("\nWin rate by exit reason:")
display(ou.groupby("exit_reason")["win"].mean().round(3))

print("\nO-U threshold width (pi+ - pi-) in spread units:")
ou["pi_width"] = ou["pi_plus"] - ou["pi_minus"]
print(ou.groupby("win")["pi_width"].describe().round(3).to_string())

print("\nEntry z-score (s0 vs m0 in std units) by outcome:")
print(ou.groupby("win")["z_entry"].describe().round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title in zip(
    axes,
    ["pi_plus_z", "pi_minus_z", "ou_sharpe"],
    ["pi+ / std at entry", "pi- / std at entry", "O-U grid Sharpe at entry"],
):
    for reason, sub in ou.groupby("exit_reason"):
        if len(sub) < 5:
            continue
        ax.hist(sub[col].dropna(), bins=20, alpha=0.5, label=reason, density=True)
    ax.set_title(title)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

print("\nNote: signed_spread_pnl is spread-space (not engine $). Equity loss also reflects costs, unfilled rebalance, overlapping pairs.")